# Pharmaceutical R&D Document Research & Critique
Agentic deep research and critique of pharmaceutical documents using Dataiku LLM Mesh + SerpAPI.

In [ ]:
# Run once to install dependencies (comment out after first run)
# %pip install -r ../requirements.txt

In [ ]:
import sys
import os

sys.path.insert(0, "..")

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

from src import ResearchCritiqueSystem, Config
from src.llm.mesh_client import MeshClient

# Load .env if running locally outside Dataiku
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

In [ ]:
# ── Discover available LLMs and project variables from Dataiku ─────────────
available_llms = MeshClient.list_llm_ids()

if not available_llms:
    available_llms = [
        "openai:gpt-4o",
        "openai:gpt-4o-mini",
        "anthropic:claude-sonnet-4-6",
        "anthropic:claude-haiku-4-5",
    ]
    print("Could not query Dataiku Mesh — using fallback LLM list.")
else:
    print(f"Found {len(available_llms)} LLMs in Mesh.")

# Pre-load serp_api_key from Dataiku project variables
_default_serp_key = ""
try:
    import dataiku
    _proj_vars = dataiku.api_client().get_project(dataiku.default_project_key()).get_variables()
    _default_serp_key = _proj_vars.get("standard", {}).get("serp_api_key", "")
    if _default_serp_key:
        print("serp_api_key loaded from project variables.")
except Exception:
    pass

In [ ]:
# ── UI Widgets ─────────────────────────────────────────────────────────────

llm_dropdown = widgets.Dropdown(
    options=available_llms,
    description="LLM:",
    layout=widgets.Layout(width="50%"),
    style={"description_width": "initial"},
)

serp_key_input = widgets.Password(
    value=_default_serp_key,
    description="SerpAPI key:",
    placeholder="Loaded from project vars — override here if needed",
    layout=widgets.Layout(width="50%"),
    style={"description_width": "initial"},
)

file_input = widgets.Text(
    description="Document path:",
    placeholder="/path/to/document.pdf  (PDF, DOCX, or TXT)",
    layout=widgets.Layout(width="80%"),
    style={"description_width": "initial"},
)

description_input = widgets.Textarea(
    description="Analysis goal:",
    placeholder=(
        "Describe what you want to analyze or critique.\n"
        "Example: Evaluate this Phase II trial report for statistical rigor, "
        "alignment with ICH E9 guidelines, and identify evidence gaps."
    ),
    layout=widgets.Layout(width="80%", height="100px"),
    style={"description_width": "initial"},
)

max_iter_slider = widgets.IntSlider(
    value=12,
    min=4,
    max=20,
    step=1,
    description="Max agent steps:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="40%"),
)

min_search_slider = widgets.IntSlider(
    value=3,
    min=1,
    max=8,
    step=1,
    description="Min web searches:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="40%"),
)

run_button = widgets.Button(
    description="Run Analysis",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(width="160px", height="36px"),
)

status_label = widgets.Label(value="")
output_area = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="10px"))

# ── Layout ─────────────────────────────────────────────────────────────────
display(
    widgets.VBox([
        widgets.HTML("<h3>Configuration</h3>"),
        llm_dropdown,
        serp_key_input,
        file_input,
        description_input,
        widgets.HBox([max_iter_slider, min_search_slider]),
        widgets.HBox([run_button, status_label]),
        output_area,
    ])
)

In [ ]:
# ── Run handler ────────────────────────────────────────────────────────────

_system_ref = {"system": None}  # holds reference for post-run access

def on_run_clicked(b):
    file_path = file_input.value.strip()
    description = description_input.value.strip()
    serp_key = serp_key_input.value.strip()

    if not file_path:
        status_label.value = "Please enter a document path."
        return
    if not description:
        status_label.value = "Please enter an analysis goal."
        return
    if not serp_key:
        status_label.value = "SerpAPI key is required."
        return

    run_button.disabled = True
    status_label.value = "Running..."

    config = Config(
        llm_id=llm_dropdown.value,
        serp_api_key=serp_key,
        max_iterations=max_iter_slider.value,
        min_searches_required=min_search_slider.value,
    )

    system = ResearchCritiqueSystem(config)
    _system_ref["system"] = system

    accumulated = []

    try:
        with output_area:
            clear_output(wait=True)
            for chunk in system.run_stream(file_path=file_path, description=description):
                accumulated.append(chunk)
                clear_output(wait=True)
                display(Markdown("".join(accumulated)))
        status_label.value = "Done."
    except Exception as e:
        with output_area:
            clear_output(wait=True)
            display(Markdown(f"**Error:** `{e}`"))
        status_label.value = f"Error: {e}"
    finally:
        run_button.disabled = False

run_button.on_click(on_run_clicked)

In [ ]:
# ── Optional: export the report after the run ──────────────────────────────
# Uncomment and run this cell after the analysis completes.

# system = _system_ref["system"]
# if system and system._last_report:
#     report = system._last_report
#     output_path = file_input.value.replace(".pdf", "_critique.md").replace(".docx", "_critique.md")
#     with open(output_path, "w") as f:
#         f.write(report.raw_markdown)
#     print(f"Report saved to: {output_path}")
# else:
#     print("Run the analysis first.")